# Data Cleaning - Laboratory Data

**Input:** `Dataset/Data/laboratory_data.csv`

**Output:** `Dataset/Clean_Data/laboratory_data_clean.csv`

---

## Vấn đề phát hiện từ EDA (III_laboratory_data.ipynb)

| # | Vấn đề | Mô tả | Mức độ ảnh hưởng |
|---|--------|-------|------------------|
| 1 | Missing values | 0 giá trị thiếu | Thấp |
| 2 | Duplicates | 0 hàng trùng lặp | Thấp |
| 3 | Biological outliers | Một số giá trị nằm ngoài khoảng sinh học hợp lý (Hemoglobin max 50, RBC max 34) | Trung bình - cần đánh dấu để review |
| 4 | Imbalanced labels | Lớp Anemia chiếm tỷ trọng cao (2,979), Cardiovascular disease thấp (697) | Trung bình - cần lưu ý khi modeling |
1. **Strip columns:** Xóa khoảng trắng thừa ở tên cột.
2. **Missing values:** Không có, giữ nguyên.
3. **Duplicates:** Không có, giữ nguyên.
4. **Outliers:** Thêm cột `outlier_flag` đánh dấu các giá trị ngoài khoảng sinh học tham khảo.
5. **Export:** Lưu kết quả ra `Clean_Data/laboratory_data_clean.csv`.

---

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================
# CONFIGURATION
# ============================================
INPUT_PATH = "../Data/laboratory_data.csv"
OUTPUT_DIR = "../Clean_Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "laboratory_data_clean.csv")

# Biological reference ranges (adult)
BIOLOGICAL_RANGES = {
    'Age': (0, 120),
    'Hemoglobin': (3, 25),
    'RBC': (1, 10),
    'WBC': (500, 100000),
    'AST (aspartate aminotransferase)': (0, 1000),
    'ALT (alanine aminotransferase)': (0, 1000),
    'Cholestrol': (50, 600),
    'Spirometry': (0, 8),
    'Creatinine': (0.1, 15),
    'Glucose': (20, 500),
    'Lipase': (0, 5000),
    'Troponin': (0, 50)
}

In [ ]:
# Đọc dữ liệu
df = pd.read_csv(INPUT_PATH)
print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(5)
# Strip whitespace from column names
df.columns = df.columns.str.strip()
print('Columns after strip:', list(df.columns))
df.head(5)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_sum = missing.sum()
print('=== Missing Values ===')
if missing_sum == 0:
    print('Không có missing values.')
else:
    print(f'Tổng missing values: {missing_sum}')
    print(missing[missing > 0])

# Duplicates
duplicates = df.duplicated().sum()
print(f'\n=== Duplicates ===')
print(f'Số hàng trùng lặp: {duplicates}')

In [ ]:
# Tạo outlier flags
outlier_flags = pd.DataFrame(index=df.index)
numeric_cols = [c for c in df.columns if c != 'Disease']

for col, (low, high) in BIOLOGICAL_RANGES.items():
    if col in df.columns:
        valid = df[col].dropna()
        outlier_mask = (valid < low) | (valid > high)
        outlier_flags[f'{col}_outlier'] = False
        outlier_flags.loc[valid.index, f'{col}_outlier'] = outlier_mask

# Count outliers per row
outlier_count = outlier_flags.sum(axis=1)
df['outlier_flag'] = (outlier_count > 0).astype(int)

print(f"Rows with at least one outlier: {df['outlier_flag'].sum()} ({df['outlier_flag'].sum()/len(df)*100:.1f}%)")
print(f"Rows without outliers: {(df['outlier_flag']==0).sum()} ({(df['outlier_flag']==0).sum()/len(df)*100:.1f}%)")

print("\n=== Top Columns by Outlier Count ===")
outlier_summary = {}
for col in outlier_flags.columns:
    cnt = outlier_flags[col].sum()
    outlier_summary[col] = cnt

for col, cnt in sorted(outlier_summary.items(), key=lambda x: -x[1])[:10]:
    print(f"  {col}: {cnt} ({cnt/len(df)*100:.1f}%)")

In [ ]:
print("=== FINAL DATA SUMMARY ===")
print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(f"Missing: {df.isnull().sum().sum()}")
print(f"\n=== Duplicates ===")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"\n=== Outlier Summary ===")
print(f"Rows with outliers: {df['outlier_flag'].sum()} ({df['outlier_flag'].sum()/len(df)*100:.1f}%)")

In [ ]:
# Export cleaned data
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print(f"Cleaned data exported to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

- **Input:** `Dataset/Data/laboratory_data.csv` - 12,009 rows × 14 cols.
- **Output:** `Dataset/Clean_Data/laboratory_data_clean.csv` - 12,009 rows × 15 cols.
- **Missing values:** 0 - không có giá trị thiếu.
- **Duplicates:** 0 - không có hàng trùng lặp.
- **Outliers:** Thêm cột `outlier_flag` đánh dấu các dòng có giá trị ngoài range sinh học.
- **Labels:** Cột `Disease` được giữ nguyên (9 lớp).

| Chỉ tiêu | Trước | Sau |
|----------|-------|-----|
| Số dòng | 12,009 | 12,009 |
| Số cột | 14 | 15 (thêm `outlier_flag`) |
| Missing | 0 | 0 |
| Duplicates | 0 | 0 |
| Outlier flag | - | Có |
| Target | Có (Disease, 9 lớp) | Có (Disease, 9 lớp) |